# Quantum Hardware Verification

**Purpose:** This notebook strictly serves to verify the trained PyTorch quantum-classical hybrid model using **real IBM Quantum hardware** for pure inference. The evaluation on physical Quantum Processing Units (QPUs) is essential to validate the robustness of the Quantum Neural Network (QNN) under real-world noise models, compared to statevector simulation.

**Constraints & Expectations:**
- 🚫 **No Training:** Pure evaluation phase. No gradients, optimizers, or weights updates (`eval()` + `torch.no_grad()`).
- ⚡ **Real Hardware Execution:** Utilizes PennyLane's `qiskit.remote` device to compile workloads and submit them to IBM Cloud infrastructure.
- 🔐 **Configuration:** Requires Colab Secrets `IBM_QUANTUM_TOKEN` for IBM Qiskit Runtime service authentication.
- 📁 **Storage:** Mounts Google Drive to load pre-trained frozen classical weights and the optimized Parameterized Quantum Circuit (PQC) parameters.

In [1]:
# Colab setup: mount Drive, clone repo, checkout main branch
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/lburdman/qnn-transfer-learning.git'
REPO_PATH = Path('/content/qnn-transfer-learning')
BRANCH = 'main'

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not REPO_PATH.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], check=True)
    os.chdir(REPO_PATH)
    subprocess.run(['git', 'fetch'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
else:
    os.chdir(Path('.'))

sys.path.append(str(Path.cwd() / 'src'))
print('Working dir:', Path.cwd())
print('Python path updated with src/')


Mounted at /content/drive
Working dir: /content/qnn-transfer-learning
Python path updated with src/


In [2]:
!pip -q install qiskit-ibm-runtime pennylane-qiskit qiskit-ibm-provider

import os
from pathlib import Path
import torch
from qiskit_ibm_runtime import QiskitRuntimeService

if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get("IBM_QUANTUM_TOKEN")
    instance_crn = userdata.get("IBM_CRN")
else:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("IBM_QUANTUM_TOKEN")
    instance_crn = os.getenv("IBM_CRN")

if not api_key:
    raise ValueError(
        "❌ IBM_QUANTUM_TOKEN no encontrado.\n"
        "Creá un Secret con ese nombre en Colab, o usa un archivo .env si corres local."
    )
else:
    print("✅ IBM Quantum API key encontrada.")

# 3) Inicializar servicio Qiskit Runtime
service = QiskitRuntimeService(
    token=api_key,
    instance=instance_crn  # si es None, Qiskit intenta resolver por defecto
)

print("✅ Conectado OK a Qiskit Runtime.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.9/249.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.7/378.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB

qiskit_runtime_service._discover_account:WARNING:2026-02-24 23:26:40,243: Loading account with the given token. A saved account will not be used.


✅ IBM Quantum API key encontrada.
✅ Conectado OK a Qiskit Runtime.


## 1. Environment & Architecture Configuration
We declare the model architecture parameters, testing dataset targets, and the quantity of quantum measurements (`SHOTS`) desired per inference trace. The classes to evaluate are dynamically pulled from the model's experimental `config.json`.

In [3]:
import json
import os

# --- CONFIGURE THESE PATHS ---
MODEL_PATH = "/content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/model.pt"
TEST_DATA_DIR = "/content/drive/MyDrive/CREMAD/Embeddings/ResNet18/val"
BATCH_SIZE = 8
SHOTS = 1024

# Read config from model folder to filter data
config_path = os.path.join(os.path.dirname(MODEL_PATH), "config.json")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config_data = json.load(f)

    if "selected_classes" in config_data:
        selected_classes = config_data["selected_classes"]
    elif "config" in config_data and "selected_classes" in config_data["config"]:
        selected_classes = config_data["config"]["selected_classes"]
    else:
        selected_classes = ['ANG', 'SAD']
else:
    selected_classes = ['ANG', 'SAD']

print(f"Loaded config from {config_path}")
print(f"Filtering dataset for classes: {selected_classes}")

from src.model_builder import build_model
from src.dataset import AudioFeatureDataset
from torch.utils.data import DataLoader
from torchvision import transforms


Loaded config from /content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/config.json
Filtering dataset for classes: ['ANG', 'SAD']


## 2. IBM Quantum Runtime Service Connectivity
Here we establish an active network session with the IBM Quantum infrastructure. By querying the `QiskitRuntimeService`, we filter out pure classical simulators and specifically request physical backend availability. The `qiskit.remote` PennyLane interface then connects the PyTorch QNode directly to the selected QPU (e.g. `ibm_fez` or `ibm_kyoto`).

In [8]:
from qiskit_ibm_runtime import QiskitRuntimeService
import pennylane as qml

# Authenticate with the provider for PennyLane backend searching
try:
    service = QiskitRuntimeService(token=api_key, channel="ibm_quantum")
except Exception as e:
    QiskitRuntimeService.save_account(token=api_key, channel="ibm_quantum_platform", overwrite=True)
    service = QiskitRuntimeService()

# Get actual backend objects
all_backends = service.backends()
real_backends = [b for b in all_backends if not b.configuration().simulator]

print("Available Real IBM Quantum Backends:")
available_backend_names = [b.name for b in real_backends]
for b_name in available_backend_names:
    print(f" - {b_name}")

if not real_backends:
    raise RuntimeError("❌ No real hardware backends currently available to your account.")

# Select the first backend object directly
selected_backend_object = real_backends[0]
BACKEND_NAME = selected_backend_object.name # Keep BACKEND_NAME for printing

print(f"Selected Backend: {BACKEND_NAME}")

n_qubits = 2 # Adjust based on your model architecture
try:
    # Pass the actual backend object to PennyLane
    dev = qml.device('qiskit.remote', wires=n_qubits, backend=selected_backend_object, shots=SHOTS)
    print(f"✅ PennyLane device connected to {BACKEND_NAME}")
except Exception as e:
    raise RuntimeError(f"❌ Failed to initialize Qiskit Remote device: {e}")

qiskit_runtime_service.__init__:WARNING:2026-02-24 23:30:24,688: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-24 23:30:24,689: Loading instance: open-instance, plan: open


Available Real IBM Quantum Backends:
 - ibm_fez
 - ibm_torino
 - ibm_marrakesh
Selected Backend: ibm_fez


/usr/local/lib/python3.12/dist-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


✅ PennyLane device connected to ibm_fez


## 3. Hybrid Model Instantiation & Topology Prep
The PyTorch environment reconstructs the Quantum-Classical transfer learning architecture. It builds the frozen classical extractor (ResNet18 embeddings) leading into the dense quantum layer (our defined Variational Quantum Circuit). The saved experimental weights are directly mapped into the state. 

In [15]:
device = torch.device("cpu") # Inference on CPU is fine since the QNode handles the heavy quantum lifting

# Reconstruct class_names based on sorted selected_classes
class_names = sorted(selected_classes)

# Using standard repo build_model logic
model_config = {
    'base_model': 'emb_resnet18',
    'quantum': True,
    'n_qubits': n_qubits,
    'q_depth': 3,
    'max_layers': 10,
    'q_delta': 0.01,
}

# Override using what we found in config if available
# The config_data dictionary is flat, so iterate directly over its items
if 'config_data' in locals():
    for k, v in config_data.items():
        # Only update/add keys that are relevant for the model_config
        if k in model_config or k == 'classical_model':
            model_config[k] = v

print("Instantiating Model...")

# --- FIX: Provide a dummy dataloader for model instantiation ---
# The build_model function needs to infer input_dim from dataloaders['train']
# For emb_resnet18, the embedding dimension is 512.
from torch.utils.data import Dataset, DataLoader
class DummyDataset(Dataset):
    def __len__(self):
        return 1
    def __getitem__(self, idx):
        # Return a dummy tensor with the expected embedding dimension (512 for ResNet18)
        return torch.randn(512), 0 # Dummy input and label

dummy_dataloader = DataLoader(DummyDataset(), batch_size=1) # batch_size doesn't matter for shape inference
dataloaders_for_build = {"train": dummy_dataloader}

model = build_model(model_config, class_names, dataloaders=dataloaders_for_build, device=device)

# LOAD WEIGHTS
if not os.path.exists(MODEL_PATH):
    print(f"❌ Could not find model at {MODEL_PATH}")
else:
    print(f"Loading weights from {MODEL_PATH}...")
    state_dict = torch.load(MODEL_PATH, map_location=device)
    if isinstance(state_dict, torch.nn.Module):
        model = state_dict
    else:
        model.load_state_dict(state_dict)

    model.to(device)
    model.eval() # CRITICAL for inference!
    print("✅ Model loaded and set to evaluation mode.")

# DATA PREPARATION (Filtering subset by reading from subfolders)
try:
    filepaths = []
    labels = []
    for i, cls_name in enumerate(class_names):
        cls_dir = os.path.join(TEST_DATA_DIR, cls_name)
        if os.path.exists(cls_dir):
            # Tomamos todos los archivos del directorio de la clase filtrada
            for file in os.listdir(cls_dir):
                filepaths.append(os.path.join(cls_dir, file))
                labels.append(i)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    test_dataset = AudioFeatureDataset(filepaths, labels, base_model=model_config['base_model'])
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)
    print(f"✅ Data loaded: {len(test_dataset)} valid samples found.")
except Exception as e:
    print(f"⚠️ Warning: Data load failed. Make sure {TEST_DATA_DIR} exists with valid samples.")
    print(e)
    test_loader = []


Instantiating Model...
Loading weights from /content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/model.pt...
✅ Model loaded and set to evaluation mode.
✅ Data loaded: 481 valid samples found.


## 4. Hardware Inference Execution
This sends our filtered dataset to IBM Quantum Hardware.

In [16]:
import time
import torch.nn.functional as F

if not test_loader:
    print("No test data found. Creating a random dummy tensor to prove hardware execution...")
    dummy_inputs = torch.randn(BATCH_SIZE, 3, 224, 224)
    dummy_labels = torch.randint(0, len(class_names), (BATCH_SIZE,))
    test_loader = [(dummy_inputs, dummy_labels)]

print(f"Executing batch of {BATCH_SIZE} samples on {BACKEND_NAME}...")
start_time = time.time()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        logits = model(inputs)

        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        print("\n--- Inference Results ---")
        for i in range(len(inputs)):
            print(f"Sample {i+1}:")
            print(f"  Predicted Class: {class_names[preds[i]]}")
            print(f"  Confidence:      {probs[i][preds[i]]:.4f}")
            if labels is not None:
                print(f"  Actual Class:    {class_names[labels[i]]}")

        break # Only run one batch to save hardware time

end_time = time.time()
print(f"\n✅ Total Hardware Execution Time: {end_time - start_time:.2f} seconds")


Executing batch of 8 samples on ibm_fez...

--- Inference Results ---
Sample 1:
  Predicted Class: SAD
  Confidence:      0.7875
  Actual Class:    SAD
Sample 2:
  Predicted Class: SAD
  Confidence:      0.9115
  Actual Class:    SAD
Sample 3:
  Predicted Class: ANG
  Confidence:      0.8806
  Actual Class:    ANG
Sample 4:
  Predicted Class: SAD
  Confidence:      0.9360
  Actual Class:    SAD
Sample 5:
  Predicted Class: ANG
  Confidence:      0.9231
  Actual Class:    ANG
Sample 6:
  Predicted Class: SAD
  Confidence:      0.9360
  Actual Class:    SAD
Sample 7:
  Predicted Class: SAD
  Confidence:      0.9265
  Actual Class:    SAD
Sample 8:
  Predicted Class: ANG
  Confidence:      0.9480
  Actual Class:    ANG

✅ Total Hardware Execution Time: 5.76 seconds


## 5. Global Validation Hardware Accuracy
The final objective is to feed the full validation subset collected from configuration (e.g., all samples within `ANG` and `SAD` splits) into the real Quantum Hardware pipeline. By accumulating the `argmax` classification traces across the entire loader, we calculate the absolute empirical accuracy of our model under real physical quantum noise characteristics.

In [ ]:
import time
import torch.nn.functional as F

print(f"Executing global evaluation over {len(test_dataset)} samples on {BACKEND_NAME}...")
start_time = time.time()

correct = 0
total = 0

with torch.no_grad():
    for batch_idx, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        logits = model(inputs)
        
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        batch_correct = (preds.cpu() == labels.cpu()).sum().item()
        correct += batch_correct
        total += labels.size(0)
        
        print(f"Batch {batch_idx+1}/{len(test_loader)} Processed | Batch Accuracy: {batch_correct/labels.size(0)*100:.1f}%")

end_time = time.time()
accuracy = 100 * correct / (total if total > 0 else 1)
print(f"\n✅ Global Hardware Execution Time: {end_time - start_time:.2f} seconds")
print(f"🎯 Global Hardware Inference Accuracy: {accuracy:.2f}% ({correct}/{total})")
